In [1]:
import pandas as pd #  pandas
import seaborn as sns # seaborn
import matplotlib.pyplot as plt # matplotlib.pyplot
from pprint import pprint # pprint para visualización metadatos
from ucimlrepo import fetch_ucirepo # se importa fetch_ucilrepo
import numpy as np # numpy
import seaborn as snst # seaborn
import statsmodels.api as sm #statsmodels
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.model_selection import GridSearchCV

# Se carga el dataset DARWIN
darwin = fetch_ucirepo(id=732)
# X (features)
x = darwin.data.features

# Y (targets)
y = darwin.data.targets

# Se concatena features y target en un mismo dataframe 'df'
df = pd.concat([x,y], axis=1)

# Análisis de ML

En este fichero se aplican los distintos algoritmos para clasificar pacientes (P) vs sanos (H).

Se pretende estudiar que clasificador es mejor y evaluar si la explicabilidad es similar en todos los casos.

## 1. Regresión logística (LR)

### 1.1. Preparación de los datos

- Se elimina la columna `ID` que no se necesita para el análisis.
- Se separan las variables predictoras (X) de la objetivo (Y)
- Se recodifica la variable objetivo `class`: paciente (P) a 0 y sano (H) a 1
- Y se dividen los datos (X e Y) en `train` 80% y `test` 20% aleatoriamente.

In [2]:
# Se elimina la columna ID
# X = df (sin 'ID' ni 'class'), axis = 1 para cols
X = df.drop(['ID', 'class'], axis = 1)

# Y contiene 'class'
Y = df['class']

# Se codficia paciente (P) -> 1 y sano (H) -> 0
le = LabelEncoder()
Y = le.fit_transform(Y)

# Se dividen los datos en 80% para train y 20% para test
X_train, X_test, Y_train, Y_test = train_test_split(X, Y, test_size=0.2, random_state=42, stratify=Y)

# Se revisa la correcta división de los datos
print(f"Datos de entrenamiento {len(X_train)}")
print(f"Datos test {len(X_test)}")

Datos de entrenamiento 139
Datos test 35


### 1.2. Estandarización de los datos

Se estandarizan los datos, pues la regresión logística es sensible a la escala, para ello se emplea `StandarScaler()` (media 0, varianza 1).

In [3]:
# StandardScaler()
scaler = StandardScaler()

# Se ajusta la estandarización con X_train
X_train_scaled = scaler.fit_transform(X_train)

# Se aplica la transformación al test
X_test_scaled = scaler.transform(X_test)

### 1.3. Entrenamiento del modelo

Se entrena el modelo con los datos de entrenamiento (`X_train_scaled` y `Y_train`)

In [4]:
# Se crea el modelo
model_LR = LogisticRegression(max_iter=1000)

# Se entrena el modelo
model_LR.fit(X_train_scaled, Y_train)

print('El modelo se ha entrenado')

El modelo se ha entrenado


### 1.4. Evaluación del modelo

Se evalua el módelo con el informe de clasificación y la matriz de confusión

In [5]:
# Se realizan las predicciones
Y_pred = model_LR.predict(X_test_scaled)

# Informe de clasificación
print("--- INFORME DE CLASIFICACIÓN ---")
print(classification_report(Y_test, Y_pred))

# Matriz de confusión
print("--- MATRIZ DE CONFUSIÓN ---")
print(confusion_matrix(Y_test, Y_pred))

# Matriz
tn, fp, fn, tp = confusion_matrix(Y_test, Y_pred).ravel()

# Métricas 
# Sensibilidad (capacidad para detectar los pacientes correctamente)
sensitivity = tp / (tp + fn)

# Especificdad (capacidad para dectectar los sanos correctamente)
specificity = tn / (tn + fp)

print(f"Sensibilidad: {sensitivity}")
print(f"Especificidad: {specificity}")

--- INFORME DE CLASIFICACIÓN ---
              precision    recall  f1-score   support

           0       0.69      0.65      0.67        17
           1       0.68      0.72      0.70        18

    accuracy                           0.69        35
   macro avg       0.69      0.68      0.68        35
weighted avg       0.69      0.69      0.69        35

--- MATRIZ DE CONFUSIÓN ---
[[11  6]
 [ 5 13]]
Sensibilidad: 0.7222222222222222
Especificidad: 0.6470588235294118


****Resumen:**
- F1-score = 0.67 para sanos (H) y 0.70 para pacientes (P).
- Accuracy = 0.69
- 11 sujetos sanos (H) han sido identificados como tal (TN).
- 13 pacientes (P) han sido identificados como tal (TP).
- 5 pacientes (P) han sido identificado como sanos (H) (FN).
- 6 sanos (H) han sido identificados como pacientes (P) (FP).

### 1.5. Mejora del modelo

Se mejora el modelo ajustando los hiperparametros.

In [9]:
"""
C 
min value 0.001 
max value 5 
step 0.005
"""
c_values = np.arange(0.001, 5.005, 0.005)

# Diccionario de parámetros
param_grid = {'C': c_values}

# Búsqueda con 5 pliegues de validación cruzada
grid_search = GridSearchCV(
    LogisticRegression(max_iter=2000), 
    param_grid, 
    cv=5, 
    scoring='accuracy',
    verbose=1
)

# Se entrena el modelo
grid_search.fit(X_train_scaled, Y_train)

# Resultados
print(f"Mejor valor de C: {grid_search.best_params_['C']}")
print(f"Mejor Accuracy en CV: {grid_search.best_score_:.4f}")

Fitting 5 folds for each of 1001 candidates, totalling 5005 fits
Mejor valor de C: 0.016
Mejor Accuracy en CV: 0.9066


## 2. Decision Tree (DT)
